# Jane Street Puzzle — droppedneuralnet

Reconstruct the original model by finding the correct permutation of 97 shuffled PyTorch piece files.

**Architecture**: 48 residual Block layers + 1 LastLayer  
- Each Block: `x_out = x + out_W @ ReLU(inp_W @ x + inp_b) + out_b`  
- LastLayer: `pred = W_L @ x + b_L`

**Runtime**: Designed for GPU (T4). Set Runtime → Change runtime type → T4 GPU before running.

In [ ]:
# Cell 1 — Mount Drive and install deps
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'pandas', 'scipy'])

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


In [ ]:
# Cell 2 — Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

DATA_CSV   = '/content/drive/MyDrive/jane_street/historical_data.csv'
PIECES_DIR = '/content/drive/MyDrive/jane_street/pieces'
CKPT_FILE  = '/content/drive/MyDrive/jane_street/hc10k_best.json'  # or None to start fresh
OUT_DIR    = '/content/drive/MyDrive/jane_street/'

# The 48 paired blocks (inp_piece_idx, out_piece_idx)
# Derived from 3-metric consensus + constrained Hungarian for 10 conflict pairs.
# Uncertain pairs (margin < 0.005): inp=16→out=82, inp=61→out=66, inp=86→out=25
PAIRS = [
    {"inp": 0,  "out": 75}, {"inp": 1,  "out": 40}, {"inp": 2,  "out": 51},
    {"inp": 3,  "out": 53}, {"inp": 4,  "out": 17}, {"inp": 5,  "out": 21},
    {"inp": 10, "out": 20}, {"inp": 13, "out": 7},  {"inp": 14, "out": 33},
    {"inp": 15, "out": 67}, {"inp": 16, "out": 82}, {"inp": 18, "out": 80},
    {"inp": 23, "out": 46}, {"inp": 27, "out": 76}, {"inp": 28, "out": 12},
    {"inp": 31, "out": 36}, {"inp": 35, "out": 24}, {"inp": 37, "out": 6},
    {"inp": 39, "out": 38}, {"inp": 41, "out": 92}, {"inp": 42, "out": 55},
    {"inp": 43, "out": 34}, {"inp": 44, "out": 83}, {"inp": 45, "out": 19},
    {"inp": 48, "out": 9},  {"inp": 49, "out": 72}, {"inp": 50, "out": 57},
    {"inp": 56, "out": 30}, {"inp": 58, "out": 78}, {"inp": 59, "out": 52},
    {"inp": 60, "out": 93}, {"inp": 61, "out": 66}, {"inp": 62, "out": 79},
    {"inp": 64, "out": 70}, {"inp": 65, "out": 22}, {"inp": 68, "out": 47},
    {"inp": 69, "out": 89}, {"inp": 73, "out": 11}, {"inp": 74, "out": 90},
    {"inp": 77, "out": 32}, {"inp": 81, "out": 8},  {"inp": 84, "out": 63},
    {"inp": 86, "out": 25}, {"inp": 87, "out": 71}, {"inp": 88, "out": 54},
    {"inp": 91, "out": 29}, {"inp": 94, "out": 96}, {"inp": 95, "out": 26},
]
print(f'{len(PAIRS)} pairs loaded')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
48 pairs loaded


In [1]:
# ── FULL PIPELINE FROM SCRATCH ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch, json, random, math, pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

PIECES_DIR = '/content/drive/MyDrive/jane_street/pieces'
DATA_CSV   = '/content/drive/MyDrive/jane_street/historical_data.csv'

# ── Data ────────────────────────────────────────────────────────────────
df   = pd.read_csv(DATA_CSV)
X_np = df[[f'measurement_{i}' for i in range(48)]].values.astype(np.float32)
y_np = df['true'].values.astype(np.float32)
X10  = torch.tensor(X_np, device=device)
y10  = torch.tensor(y_np, device=device)

# ── LastLayer ────────────────────────────────────────────────────────────
d85 = torch.load(f'{PIECES_DIR}/piece_85.pth', map_location=device, weights_only=True)
W_L = d85['weight'].squeeze()   # (48,)
b_L = d85['bias'].squeeze()     # scalar

# ── Piece inventory ──────────────────────────────────────────────────────
inp_pieces = [0,1,2,3,4,5,10,13,14,15,16,18,23,27,28,31,35,37,39,41,
              42,43,44,45,48,49,50,56,58,59,60,61,62,64,65,68,69,73,
              74,77,81,84,86,87,88,91,94,95]
out_pieces = [6,7,8,9,11,12,17,19,20,21,22,24,25,26,29,30,32,33,34,36,
              38,40,46,47,51,52,53,54,55,57,63,66,67,70,71,72,75,76,78,
              79,80,82,83,89,90,92,93,96]

# ── Hungarian pairing ────────────────────────────────────────────────────
all_inp, all_out = {}, {}
for ip in inp_pieces:
    d = torch.load(f'{PIECES_DIR}/piece_{ip}.pth', map_location=device, weights_only=True)
    all_inp[ip] = (d['weight'], d['bias'])
for op in out_pieces:
    d = torch.load(f'{PIECES_DIR}/piece_{op}.pth', map_location=device, weights_only=True)
    all_out[op] = (d['weight'], d['bias'])

score_matrix = torch.zeros(48, 48)
for i, ip in enumerate(inp_pieces):
    for j, op in enumerate(out_pieces):
        P = all_out[op][0] @ all_inp[ip][0]
        score_matrix[i,j] = torch.abs(torch.trace(P)) / torch.norm(P, 'fro')

row_ind, col_ind = linear_sum_assignment(-score_matrix.numpy())
pairs = [(inp_pieces[i], out_pieces[j]) for i,j in zip(row_ind, col_ind)]
print(f"Paired {len(pairs)} blocks via Hungarian")

# ── Load blocks ──────────────────────────────────────────────────────────
NB = 48
blocks = []
for ip, op in pairs:
    iW, ib = all_inp[ip]
    oW, ob = all_out[op]
    blocks.append((ip, op, iW.t().contiguous(), ib, oW.t().contiguous(), ob))

# ── Forward pass & scoring ───────────────────────────────────────────────
@torch.no_grad()
def forward(order, Xb):
    x = Xb.clone()
    for bi in order:
        _, _, iW_T, ib, oW_T, ob = blocks[bi]
        h = x.mm(iW_T).add_(ib).clamp_(min=0)
        x = x.add(h.mm(oW_T).add_(ob))
    return x

@torch.no_grad()
def mse_10k(order):
    out = forward(order, X10).mv(W_L).add_(b_L)
    return float(((out - y10)**2).mean())

@torch.no_grad()
def r2_10k(order):
    out = forward(order, X10).mv(W_L).add_(b_L)
    ss_r = float(((out - y10)**2).sum())
    ss_t = float(((y10 - y10.mean())**2).sum())
    return 1 - ss_r/ss_t

# ── Benchmark ────────────────────────────────────────────────────────────
import time
t0 = time.time()
mse_10k(list(range(NB)))
print(f"Benchmark: {(time.time()-t0)*1000:.1f}ms per 10k pass")
print("Ready.")

Mounted at /content/drive
Device: cuda
Paired 48 blocks via Hungarian
Benchmark: 239.1ms per 10k pass
Ready.


In [ ]:
# ── SA + Hill Climb ──────────────────────────────────────────────────────
import random, math, time

best_overall_m = float('inf')
best_overall_ord = None

for trial in range(20):
    random.seed(trial * 17 + 3)
    order = list(range(NB))
    random.shuffle(order)
    cur = mse_10k(order)
    best_m, best_ord = cur, order[:]
    T = 3.0

    for it in range(30000):
        new_order = order[:]
        r = random.randint(0, 2)
        if r == 0:
            i, j = random.sample(range(NB), 2)
            new_order[i], new_order[j] = new_order[j], new_order[i]
        elif r == 1:
            i = random.randrange(NB)
            j = random.randrange(NB)
            elem = new_order.pop(i)
            new_order.insert(j, elem)
        else:
            i, j = sorted(random.sample(range(NB), 2))
            new_order[i:j] = new_order[i:j][::-1]

        nm = mse_10k(new_order)
        d = nm - cur
        if d < 0 or random.random() < math.exp(-d / max(T, 1e-10)):
            order, cur = new_order, nm
            if cur < best_m:
                best_m, best_ord = cur, order[:]
        T *= 0.9998

    # hill climb: swap + insert
    order = best_ord[:]
    cur = best_m
    for rnd in range(200):
        improved = False
        for i in range(NB):
            for j in range(i+1, NB):
                order[i], order[j] = order[j], order[i]
                m = mse_10k(order)
                if m < cur - 1e-6:
                    cur = m
                    improved = True
                else:
                    order[i], order[j] = order[j], order[i]
        for i in range(NB):
            elem = order.pop(i)
            best_j, best_m_ins = i, cur
            for j in range(NB):
                if j == i: continue
                order.insert(j, elem)
                m = mse_10k(order)
                if m < best_m_ins - 1e-6:
                    best_m_ins, best_j = m, j
                order.pop(j)
            order.insert(best_j, elem)
            if best_j != i:
                cur = best_m_ins
                improved = True
        if not improved:
            break

    if cur < best_overall_m:
        best_overall_m = cur
        best_overall_ord = order[:]

    print(f"Trial {trial+1:>2}: MSE={cur:.6f}  R²={r2_10k(order):.4f}  overall_best={best_overall_m:.6f}")

# Build solution
solution = []
for bi in best_overall_ord:
    b = blocks[bi]
    solution.append(int(b[0]))
    solution.append(int(b[1]))
solution.append(85)
assert len(solution) == 97 and len(set(solution)) == 97
print(f"\nBest: MSE={best_overall_m:.6f}  R²={r2_10k(best_overall_ord):.4f}")
print(f"Solution: {solution}")


Trial  1: MSE=0.090635  R²=0.9013  overall_best=0.090635
Trial  2: MSE=0.090164  R²=0.9018  overall_best=0.090164
Trial  3: MSE=0.090193  R²=0.9018  overall_best=0.090164
Trial  4: MSE=0.090164  R²=0.9018  overall_best=0.090164
Trial  5: MSE=0.090728  R²=0.9012  overall_best=0.090164
Trial  6: MSE=0.090407  R²=0.9016  overall_best=0.090164
Trial  7: MSE=0.091753  R²=0.9001  overall_best=0.090164
Trial  8: MSE=0.090635  R²=0.9013  overall_best=0.090164
Trial  9: MSE=0.090149  R²=0.9019  overall_best=0.090149
Trial 10: MSE=0.090188  R²=0.9018  overall_best=0.090149
Trial 11: MSE=0.089985  R²=0.9020  overall_best=0.089985
Trial 12: MSE=0.090270  R²=0.9017  overall_best=0.089985
Trial 13: MSE=0.090164  R²=0.9018  overall_best=0.089985
Trial 14: MSE=0.090255  R²=0.9017  overall_best=0.089985
Trial 15: MSE=0.090164  R²=0.9018  overall_best=0.089985
Trial 16: MSE=0.089992  R²=0.9020  overall_best=0.089985


In [ ]:
# Cell 3 — Load data and pieces onto GPU
import torch, json, numpy as np, pandas as pd, time, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

df   = pd.read_csv(DATA_CSV)
X_np = df[[f'measurement_{i}' for i in range(48)]].values.astype(np.float32)
y_np = df['true'].values.astype(np.float32)
N    = len(X_np)
print(f'Data: {N} rows × {X_np.shape[1]} features')

# 2k-row subsample for candidate selection
rng0   = np.random.default_rng(7)
s2k    = rng0.choice(N, 2000, replace=False)
X2k_np = np.ascontiguousarray(X_np[s2k])
y2k_np = y_np[s2k]

# Move to GPU
X10 = torch.from_numpy(X_np).to(device)
y10 = torch.from_numpy(y_np).to(device)
X2k = torch.from_numpy(X2k_np).to(device)
y2k = torch.from_numpy(y2k_np).to(device)

# LastLayer (piece_85)
d85 = torch.load(f'{PIECES_DIR}/piece_85.pth', map_location='cpu', weights_only=True)
W_L = d85['weight'].float().squeeze(0).to(device)   # (48,)
b_L = float(d85['bias'])

# Blocks: pre-transpose weights for x @ iW_T = (N,48)@(48,96)=(N,96)
blocks = []
for p in PAIRS:
    ip, op = p['inp'], p['out']
    di = torch.load(f'{PIECES_DIR}/piece_{ip}.pth', map_location='cpu', weights_only=True)
    do = torch.load(f'{PIECES_DIR}/piece_{op}.pth', map_location='cpu', weights_only=True)
    iW_T = di['weight'].float().T.contiguous().to(device)   # (48, 96)
    ib   = di['bias'].float().to(device)                    # (96,)
    oW_T = do['weight'].float().T.contiguous().to(device)   # (96, 48)
    ob   = do['bias'].float().to(device)                    # (48,)
    blocks.append((ip, op, iW_T, ib, oW_T, ob))

NB = len(blocks)
print(f'{NB} blocks loaded to {device}')

# Benchmark
order0 = list(range(NB))
@torch.no_grad()
def forward_t(x, order):
    x = x.clone()
    for bi in order:
        _, _, iW_T, ib, oW_T, ob = blocks[bi]
        h = x.mm(iW_T).add_(ib).clamp_(min=0)
        x = x.add(h.mm(oW_T).add_(ob))
    return x

if device.type == 'cuda': torch.cuda.synchronize()
t0 = time.time()
for _ in range(5): forward_t(X10, order0)
if device.type == 'cuda': torch.cuda.synchronize()
ms10 = (time.time()-t0)/5*1000

t0 = time.time()
for _ in range(5): forward_t(X2k, order0)
if device.type == 'cuda': torch.cuda.synchronize()
ms2k = (time.time()-t0)/5*1000
print(f'Benchmark: 10k={ms10:.1f}ms/pass  2k={ms2k:.1f}ms/pass')

Using device: cuda
Data: 10000 rows × 48 features
48 blocks loaded to cuda
Benchmark: 10k=53.7ms/pass  2k=4.1ms/pass


In [ ]:
# Cell 4 — MSE/R² helpers and state cache
@torch.no_grad()
def mse_t(order, Xb=None, yb=None):
    if Xb is None: Xb, yb = X2k, y2k
    xf   = forward_t(Xb, order)
    pred = xf.mv(W_L).add_(b_L)
    diff = pred.sub(yb)
    return float(diff.dot(diff)) / len(yb)

@torch.no_grad()
def mse_10k(order): return mse_t(order, X10, y10)

@torch.no_grad()
def r2_10k(order):
    xf   = forward_t(X10, order)
    pred = xf.mv(W_L).add_(b_L)
    diff = pred.sub(y10)
    ss_r = float(diff.dot(diff))
    ss_t = float((y10 - y10.mean()).pow(2).sum())
    return 1.0 - ss_r / ss_t

@torch.no_grad()
def compute_states(order, Xb=None):
    if Xb is None: Xb = X2k
    states = [Xb.clone()]
    x = Xb.clone()
    for bi in order:
        _, _, iW_T, ib, oW_T, ob = blocks[bi]
        h = x.mm(iW_T).add_(ib).clamp_(min=0)
        x = x.add(h.mm(oW_T).add_(ob))
        states.append(x.clone())
    return states

@torch.no_grad()
def rerun_mse_t(states, order, start, yb=None):
    if yb is None: yb = y2k
    x = states[start].clone()
    for k in range(start, NB):
        _, _, iW_T, ib, oW_T, ob = blocks[order[k]]
        h = x.mm(iW_T).add_(ib).clamp_(min=0)
        x = x.add(h.mm(oW_T).add_(ob))
    pred = x.mv(W_L).add_(b_L)
    diff = pred - yb
    return float(diff.dot(diff)) / len(yb)

def refresh_states(states, order, start):
    x = states[start].clone()
    for k in range(start, NB):
        _, _, iW_T, ib, oW_T, ob = blocks[order[k]]
        h = x.mm(iW_T).add_(ib).clamp_(min=0)
        x = x.add(h.mm(oW_T).add_(ob))
        states[k+1] = x.clone()

print('Helpers defined.')

Helpers defined.


In [ ]:
# Cell 5 — Load checkpoint and run Hill Climb (2k-select + 10k-verify)
import math

# Load starting order
if CKPT_FILE and os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        ckpt_inp = json.load(f)
    inp_to_bi = {blocks[bi][0]: bi for bi in range(NB)}
    order = [inp_to_bi[ip] for ip in ckpt_inp]
    print(f'Loaded checkpoint: {CKPT_FILE}')
else:
    # Fresh start: random order
    rng_s = np.random.default_rng(42)
    order = rng_s.permutation(NB).tolist()
    print('Starting from random order')

start_mse = mse_10k(order)
start_r2  = r2_10k(order)
print(f'Starting: 10k MSE={start_mse:.6f}  R²={start_r2:.4f}')

states_2k  = compute_states(order)
cur_2k     = rerun_mse_t(states_2k, order, 0)
cur_10k    = start_mse
best_10k   = cur_10k
best_order = order[:]

best_path = f'{OUT_DIR}/hc_gpu_best.json'
with open(best_path, 'w') as f:
    json.dump([blocks[bi][0] for bi in order], f)

t_start = time.time()
print('\n=== Hill Climb: 2k-select + 10k-verify, swap + insert ===')

for rnd in range(200):
    improved = False
    rnd_t0   = time.time()
    n_swap = n_ins = 0

    # ── swap pass ────────────────────────────────────────────────────────
    for i in range(NB):
        for j in range(i+1, NB):
            order[i], order[j] = order[j], order[i]
            cand_2k = rerun_mse_t(states_2k, order, i)
            if cand_2k < cur_2k - 1e-9:
                full_mse = mse_10k(order)
                if full_mse < cur_10k - 1e-9:
                    cur_2k  = cand_2k
                    cur_10k = full_mse
                    refresh_states(states_2k, order, i)
                    improved = True
                    n_swap  += 1
                    if cur_10k < best_10k:
                        best_10k   = cur_10k
                        best_order = order[:]
                        with open(best_path, 'w') as f:
                            json.dump([blocks[bi][0] for bi in order], f)
                else:
                    order[i], order[j] = order[j], order[i]
            else:
                order[i], order[j] = order[j], order[i]

    # ── insert pass ──────────────────────────────────────────────────────
    for i in range(NB):
        elem = order.pop(i)
        best_j, best_2k, best_full = i, cur_2k, cur_10k
        for j in range(NB):
            if j == i: continue
            order.insert(j, elem)
            cand_2k = rerun_mse_t(states_2k, order, min(i, j))
            if cand_2k < best_2k - 1e-9:
                best_2k = cand_2k
                best_j  = j
            order.pop(j)
        # verify winner on 10k
        if best_j != i:
            order.insert(best_j, elem)
            full_mse = mse_10k(order)
            if full_mse < best_full - 1e-9:
                cur_2k  = best_2k
                cur_10k = full_mse
                refresh_states(states_2k, order, min(i, best_j))
                improved = True
                n_ins   += 1
                if cur_10k < best_10k:
                    best_10k   = cur_10k
                    best_order = order[:]
                    with open(best_path, 'w') as f:
                        json.dump([blocks[bi][0] for bi in order], f)
            else:
                order.pop(best_j)
                order.insert(i, elem)
        else:
            order.insert(i, elem)

    r2_now  = r2_10k(order)
    elapsed = time.time() - t_start
    rnd_t   = time.time() - rnd_t0
    print(f'  round {rnd+1:>3}: 10k MSE={cur_10k:.6f}  R²={r2_now:.4f}  '
          f'swaps={n_swap}  ins={n_ins}  '
          f'improved={improved}  ({rnd_t:.0f}s/round, {elapsed:.0f}s total)')

    if not improved:
        break

print(f'\nBest 10k MSE={best_10k:.6f}  R²={r2_10k(best_order):.4f}')

Loaded checkpoint: /content/drive/MyDrive/jane_street/hc10k_best.json
Starting: 10k MSE=0.096103  R²=0.8954

=== Hill Climb: 2k-select + 10k-verify, swap + insert ===
  round   1: 10k MSE=0.095000  R²=0.8966  swaps=3  ins=6  improved=True  (9s/round, 9s total)
  round   2: 10k MSE=0.094770  R²=0.8968  swaps=1  ins=0  improved=True  (10s/round, 18s total)
  round   3: 10k MSE=0.094770  R²=0.8968  swaps=0  ins=0  improved=False  (10s/round, 28s total)

Best 10k MSE=0.094770  R²=0.8968


# Cell 5.1 Pairing audit — find wrong pairs
import itertools

current_mse = 0.094770
results = []

for bi in range(NB):
    inp_idx = blocks_data[bi][0]
    cur_out = blocks_data[bi][1]
    
    for bj in range(NB):
        if bi == bj: continue
        swap_out = blocks_data[bj][1]
        
        # try swapping out pieces between bi and bj
        test_blocks = list(blocks_data)
        test_blocks[bi] = (inp_idx, swap_out, *load_piece(swap_out))
        test_blocks[bj] = (blocks_data[bj][0], cur_out, *load_piece(cur_out))
        
        m = mse_10k(best_order, test_blocks)
        if current_mse - m > 0.002:
            results.append((current_mse - m, bi, bj, inp_idx, cur_out, swap_out))

results.sort(reverse=True)
print("Top 10 impactful swaps:")
for improvement, bi, bj, inp, old_out, new_out in results[:10]:
    print(f"  swap inp={inp} out {old_out}→{new_out}: improvement={improvement:.4f}")

In [ ]:
# Cell 6 — Uncertain pair validation
# For 3 pairs with small R² margins, try alternative out-pieces and keep the best.

UNCERTAIN = [
    (16, 82, [83, 20, 52, 36, 80, 24, 25, 57, 66]),
    (61, 66, [80, 70, 20, 82, 83, 24, 25, 52, 57]),
    (86, 25, [36, 57, 66, 80, 20, 82, 83, 24, 52]),
]

def replace_out_blocks(blks, inp_idx, new_out_idx):
    """Return a new blocks list with one pair's out-piece swapped."""
    do   = torch.load(f'{PIECES_DIR}/piece_{new_out_idx}.pth', map_location='cpu', weights_only=True)
    oW_T = do['weight'].float().T.contiguous().to(device)
    ob   = do['bias'].float().to(device)
    return [(b[0], new_out_idx, b[2], b[3], oW_T, ob) if b[0] == inp_idx else b
            for b in blks]

@torch.no_grad()
def score_blocks(ord_, blks):
    x = X10.clone()
    for bi in ord_:
        _, _, iW_T, ib, oW_T, ob = blks[bi]
        h = x.mm(iW_T).add_(ib).clamp_(min=0)
        x = x.add(h.mm(oW_T).add_(ob))
    pred = x.mv(W_L).add_(b_L)
    diff = pred - y10
    return float(diff.dot(diff)) / N

final_blocks = list(blocks)
final_order  = best_order[:]
base_mse     = score_blocks(final_order, final_blocks)
print(f'Base MSE before uncertain check: {base_mse:.6f}')

for inp_idx, cur_out, alts in UNCERTAIN:
    print(f'\n  inp={inp_idx}  cur_out={cur_out}')
    best_out, best_m = cur_out, base_mse
    taken = {b[1] for b in final_blocks if b[0] != inp_idx}
    for alt in alts:
        if alt in taken:
            print(f'    out={alt}: taken'); continue
        alt_blks = replace_out_blocks(final_blocks, inp_idx, alt)
        am = score_blocks(final_order, alt_blks)
        flag = ' *** BETTER' if am < best_m - 1e-6 else ''
        print(f'    out={alt}: MSE={am:.6f}{flag}')
        if am < best_m - 1e-6:
            best_m, best_out = am, alt
    if best_out != cur_out:
        print(f'  => Switching inp={inp_idx}: out {cur_out} -> {best_out}')
        final_blocks = replace_out_blocks(final_blocks, inp_idx, best_out)
        base_mse = best_m
    else:
        print(f'  => Keeping out={cur_out}')

print(f'\nAfter uncertain check: MSE={base_mse:.6f}')

In [ ]:
# Cell 7 — Write solution.json and final scoring
import os, json

out_map = {b[0]: b[1] for b in final_blocks}

solution = []
for bi in final_order:
    ip = final_blocks[bi][0]
    solution.append(int(ip))
    solution.append(int(out_map[ip]))
solution.append(85)   # LastLayer always last

assert len(solution) == 97, f'Expected 97, got {len(solution)}'
dupes = [x for x in set(solution) if solution.count(x) > 1]
assert not dupes, f'Duplicates: {dupes}'

sol_path = f'{OUT_DIR}/solution.json'
with open(sol_path, 'w') as f:
    json.dump(solution, f)
print(f'solution.json written to {sol_path}')

# Final scoring
@torch.no_grad()
def final_score(ord_, blks):
    x = X10.clone()
    for bi in ord_:
        _, _, iW_T, ib, oW_T, ob = blks[bi]
        h = x.mm(iW_T).add_(ib).clamp_(min=0)
        x = x.add(h.mm(oW_T).add_(ob))
    pred = x.mv(W_L).add_(b_L)
    diff = pred - y10
    ss_r = float(diff.dot(diff))
    ss_t = float((y10 - y10.mean()).pow(2).sum())
    mse  = ss_r / N
    r2   = 1.0 - ss_r / ss_t
    return mse, r2

final_mse, final_r2 = final_score(final_order, final_blocks)
print('=' * 60)
print(f'FINAL MSE  (all 10k): {final_mse:.6f}')
print(f'FINAL R²   (all 10k): {final_r2:.4f}')
print('=' * 60)
print(f'Solution ({len(solution)} entries): {solution}')